## DINOv2 LSTM ##

In [1]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix


import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import Dataset, DataLoader


import os
import pickle
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from PIL import Image

/media/osero/SamsungSSD/miniconda_files/conda/envs/dinov2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device

In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# device = torch.device("cpu")
device

device(type='cuda')

## Prepare Dataset

In [3]:
# Define transform to match the input size for the model
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [4]:
import pandas as pd

# Define the function to check if a class is two-handed
def is_two_handed(class_id):
    # Load the dataset
    df = pd.read_csv('/home/osero/Downloads/classes_extra.csv')
    
    # Search for the class by name
    class_row = df[df['ClassID'] == class_id] ## Labels start with 0 but id starts with 1
    if not class_row.empty:
        return bool(class_row['Two Hand'].values[0])
    else:
        raise ValueError(f"Class '{class_id}' not found in the dataset.")


In [5]:
# pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/0001/User_2_001.pickle'
pose_pickle_folder = '/media/osero/SamsungSSD/CMPE_SSD/mmpose-full/'

def get_active_frames_from_pickle(input_raw) -> np.ndarray:
    threshold = (
        (((input_raw["pose"]["left_hip"][:, 1] + input_raw["pose"]["right_hip"][:, 1]) / 2 )* 7)
        + input_raw["pose"]["nose"][:, 1] * 3
    ) / 10

    active_frames = (
        np.minimum(
            input_raw["hand_left"]["left_lunate_bone"][:, 1],
            input_raw["hand_right"]["right_lunate_bone"][:, 1],
        )
        < threshold
    )

    active_frame_indices = np.argwhere(active_frames).squeeze()
    return active_frame_indices


def get_active_frames(label_name, sample_name):
    pickle_file_name = f"{pose_pickle_folder}/{label_name}/{sample_name}.pickle"
    file = open(pickle_file_name, 'rb')
    input_raw = pickle.load(file)

    return get_active_frames_from_pickle(input_raw)

In [6]:
from torch.utils.data import Dataset, DataLoader

frame_frequency = 2

def create_label_dict(classes):
    label_dict = {}
    for i in range(0,len(classes)):
        label_dict[classes[i]] = i
    return label_dict

class CustomImageDataset(Dataset):
    def __init__(self, video_folder_left = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_left-c256", video_folder_right = "/media/osero/SamsungSSD/CMPE_SSD/frame-hand_right-c256"):

        image_file_list = []
        labels = []
        for label_folder in sorted(os.listdir(video_folder_left)):
            full_label_folder = os.path.join(video_folder_left, label_folder)
            label = int(label_folder)
            for sample_folder in sorted(os.listdir(full_label_folder)):
                full_sample_folder = os.path.join(full_label_folder, sample_folder)
                full_sample_folder_files = sorted(os.listdir(full_sample_folder))
                splited_paths = full_sample_folder.split('/')
                # if ('_001' in splited_paths[-1]):
                active_frame_indices = get_active_frames(splited_paths[-2], splited_paths[-1])
                active_frame_indices = (
                    active_frame_indices
                    if active_frame_indices.size > 10
                    else np.arange(0, len(full_sample_folder_files))
                )

                for i, image_file in enumerate(full_sample_folder_files):
                    if (i in active_frame_indices):
                        full_image_file = os.path.join(full_sample_folder, image_file)
                        image_file_list.append(full_image_file)
                        labels.append(label)

        for label_folder in sorted(os.listdir(video_folder_right)):
            full_label_folder = os.path.join(video_folder_right, label_folder)
            label = int(label_folder)
            if (is_two_handed(label)):
                for sample_folder in sorted(os.listdir(full_label_folder)):
                    full_sample_folder = os.path.join(full_label_folder, sample_folder)
                    full_sample_folder_files = sorted(os.listdir(full_sample_folder))
                    splited_paths = full_sample_folder.split('/')
                    # if ('_001' in splited_paths[-1]):
                    active_frame_indices = get_active_frames(splited_paths[-2], splited_paths[-1])
                    active_frame_indices = (
                        active_frame_indices
                        if active_frame_indices.size > 10
                        else np.arange(0, len(full_sample_folder_files))
                    )

                    for i, image_file in enumerate(full_sample_folder_files):
                        if (i in active_frame_indices):
                            full_image_file = os.path.join(full_sample_folder, image_file)
                            image_file_list.append(full_image_file)
                            labels.append(label)

        self.classes = np.unique(labels)
        label_dict = create_label_dict(self.classes)
        self.labels = [label_dict[x] for x in labels]
        self.image_file_list = image_file_list
        self.train_indices = [i for i, string in enumerate(image_file_list) if not 'user_4' in string.lower()]
        self.test_indices = [i for i, string in enumerate(image_file_list) if 'user_4' in string.lower() and '_001/' in string]

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):    

        image = Image.open(self.image_file_list[idx])
        input_tensor = transform(image)  # Add batch dimension
        return input_tensor, self.labels[idx] 


In [7]:
from torch.utils.data import Dataset, DataLoader, Subset

mixed_dataset = CustomImageDataset()

train_dataset = Subset(mixed_dataset, mixed_dataset.train_indices)
test_dataset = Subset(mixed_dataset, mixed_dataset.test_indices)

batch_size = 1
num_workers = 4

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)  # Adjust batch size as needed

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [8]:
class_names = mixed_dataset.classes
class_names

num_classes = len(set(class_names))
print("num_classes: ", num_classes)
print("train_dataset size: ", len(train_dataset))
print("test_dataset size: ", len(test_dataset))

num_classes:  744
train_dataset size:  1342766
test_dataset size:  56349


## Model

In [9]:
class VideoClassifierLSTM(nn.Module):
    def __init__(self, num_classes):
        super(VideoClassifierLSTM, self).__init__()
        self.dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitl14')
        self.fc = nn.Linear(self.dino_model.embed_dim, num_classes)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):        
        dino_feature = self.dino_model(x)
        output = self.dropout(dino_feature)
        output = self.fc(output)  # Take hidden state of the last LSTM layer
        return output
    
model = VideoClassifierLSTM(num_classes=num_classes)

# Load the state_dict into the model
# model.load_state_dict(torch.load("/home/osero/Desktop/CMPE/dinov2/classsification/lstm/lstm_results/FINE_TUNED_LEFT_MODEL2024-12-29_23-00-38_1.pth"))

model = model.to(device)

Using cache found in /home/osero/.cache/torch/hub/facebookresearch_dinov2_main
A matching Triton is not available, some optimizations will not be enabled.
Error caught was: No module named 'triton'
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/home/osero/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


## Functions

In [10]:
def test_images():
    correct = 0
    top_5_correct = 0
    total = 0
    running_loss = 0.0
    # since we're not training, we don't need to calculate the gradients for our outputs
    test_predicted = []
    test_labels = []

    with torch.no_grad():
        for image, labels in test_loader:
            image = image.to(device)
            # features = features.to(device)
            labels = labels.to(device)

            # calculate outputs by running images through the network
            outputs = model(image)
            loss = criterion(outputs, labels)
            
            # the class with the highest energy is what we choose as prediction
            _, predicted = torch.topk(outputs.data, 1)
            _, predicted_top_5 = torch.topk(outputs.data, 5)
            total += labels.size(0)
            correct += (predicted.flatten()== labels.flatten()).sum().item() 
            top_5_correct += sum([(predicted_top_5[i] == labels[i]).any().item() for i in range(len(labels))])
            running_loss += loss.item()

            test_labels += (labels.cpu().numpy().tolist())
            test_predicted += (predicted.cpu().numpy().tolist())

    avg_loss = running_loss / total
    accuracy = 100 * correct / total
    top_5_accuracy = 100 * top_5_correct / total
    print(f'Accuracy of the network on the {len(test_loader)*batch_size} test video: {accuracy:.4f} %, top5: {top_5_accuracy:.4f} %, avg_loss: {avg_loss}')
    return accuracy, top_5_accuracy, avg_loss

In [11]:
import datetime
from time import gmtime, strftime
def get_current_time():
    return strftime("%Y-%m-%d_%H-%M-%S", gmtime())

def save_model_result(current_time, epoch):
    result_name = 'lstm_results/FINE_TUNED_LEFT_LARGE' + current_time + '_' + str(epoch) + '.pth'
    result_name_MODEL = 'lstm_results/FINE_TUNED_LEFT_MODEL_LARGE' + current_time +  '_' + str(epoch) + '.pth'
    torch.save(model.state_dict(), result_name_MODEL)

    torch.save({'name': result_name,
                'model_state_dict': model.state_dict(),
                'batch_size': batch_size,
                'frame_frequency': frame_frequency,
                'num_classes': num_classes,
                'train_dataset': len(train_dataset),
                'test_dataset': len(test_dataset)},
                result_name)



## Train

In [ ]:
lr = 0.0002
step_size = 10
gamma = 0.5
weight_decay = 0

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=lr)
scheduler = lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=gamma) ## CosineAnnealingLR Dene
print(f"lr {lr}, step_size: {step_size}, gamma: {gamma}, weight_decay: {weight_decay}")
print(f"batch_size {batch_size}, frame_frequency: {frame_frequency}")

avg_loss_list = []
avg_accuracy_list = []
avg_test_accuracy_list = []
avg_top5_test_accuracy_list = []
avg_test_loss_list = []

num_epoch = 5
for epoch in range(num_epoch):
    train_acc = 0
    train_loss = 0
    loop = tqdm(train_loader)

    running_loss = 0.0
    running_accuracy= 0.0
    for idx, (image, labels) in enumerate(loop):
        image = image.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(image)
        loss = criterion(outputs, labels)

        predictions = outputs.argmax(dim=1, keepdim=True).squeeze()
        correct = (predictions == labels).sum().item()
        accuracy = correct / batch_size

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_accuracy += 100 * accuracy
        loop.set_description(f"Epoch [{epoch}/{num_epoch}]")
        loop.set_postfix(loss=loss.item(), acc=accuracy)
    scheduler.step()
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f"Time: {get_current_time()} Epoch [{epoch}], Avg loss: {avg_loss:.4f}, Avg accuracy: {avg_accuracy:.4f}")
    avg_test_accuracy, avg_top5_test_accuracy, avg_test_loss = test_images()

    avg_loss_list.append(avg_loss)
    avg_accuracy_list.append(avg_accuracy)
    avg_test_accuracy_list.append(avg_test_accuracy)
    avg_top5_test_accuracy_list.append(avg_top5_test_accuracy)
    avg_test_loss_list.append(avg_test_loss)
    current_time = get_current_time()
    save_model_result(current_time, epoch)


lr 0.0002, step_size: 10, gamma: 0.5, weight_decay: 0
batch_size 1, frame_frequency: 2


  0%|          | 0/1342766 [00:00<?, ?it/s]